# AMADS Coding Notebooks
## Count Chords

---

How many distinct chords are there in 12 TET?
A demonstration-exercise.

**By (author/s):** Mark Gotham

**For:** Attached to the
[AMADS code library](https://github.com/music-computing/amads/) and
["Keeping Score" book](https://doi.org/10.5281/zenodo.14938027),
but open to all.

**Licence:** MIT.

**Colour key:**
- <font color='green'> Green is for a block of information.
- <font color='purple'> Purple is for an exercise.
- <font color='crimson'> Crimson is for the solution to the exercise above it.

**Learning Objectives:** After completing this notebook, you will ...
- Understand how many combinations exist within 12-TET.
- Have a sense of the transformations useful to compare musical chords.
- Have a sense of which transformations are more / less musically meaningful.


In [ ]:
# (Install. if needed. Uncomment and run the line below)
# %pip install AMADS

In [ ]:
from itertools import combinations

## <font color='Green'> Information

How many distinct chords are there in 12 TET?

Let's tackle this question in "three stages and an appendix".

First we'll consider the 12 distinct pitch classes 0–12 (or C-B if you prefer, but not considering spelling).

Overall, there should be 2 ** 12 = 4096, distributed by cardinality as follows, set out to highlight the symmetry:
- 0: 1
    - 1: 12,
        - 2: 66,
            - 3: 220,
                - 4: 495,
                    - 5: 792,
                        - 6: 924,
                    - 7: 792,
                - 8: 495,
            - 9: 220,
        - 10: 66,
    - 11: 12
- 12: 1

Exercises:
1. Ignoring any further equivalences, create the set of 4096 outlined above,
    - organise into a dict with keys as cardinalities and values as a list of pitch class lists.
2. Filter for transposition equivalence.
    - for this, we will need to map pitch classes to intervals and to test rotation equivalence.
3. Further filter for inversion equivalence.

"Appendix Exercise":
- Move beyond pitch class equivalence to consider all distinct versions of one "chord" in different octaves.
- Use the strikingly simple case of one pitch class and see the large number of possiblilties!


## <font color='purple'> Exercise 1: Create the 4096

## <font color='crimson'> Solution 1

In [ ]:
def get_all_4096() -> dict:
    all_4096 = dict()
    for cardinality in range(0, 13):
        all_4096[cardinality] = list(combinations(range(12), cardinality))
    return all_4096

In [ ]:
all_4096 = get_all_4096()

In [ ]:
# E.g., here are the 2 note chords ("dyads")
all_4096[2]

## <font color='purple'> Exercise 2: Transposition Equivalence

## <font color='crimson'> Solution 2

In [ ]:
from amads.core.vector_transforms_checks import is_rotation_equivalent, indices_to_interval_sequence

def filter_transposition_equivalence(all_4096: dict) -> dict:
    """
    Introducing transposition equivalence reduces the 4096 total to 352, distributed:
    [1, 1, 6, 19, 43, 66, 80, 66, 43, 19, 6, 1, 1]
    """
    assert sum([len(v) for v in all_4096.values()]) == 4096

    transposition_equiv = dict()

    # Hard code special case of cardinalities 0, 1
    transposition_equiv[0] = [None]
    transposition_equiv[1] = [None]
    for cardinality in range(2, 13):
        transposition_equiv[cardinality] = []
        for comb in all_4096[cardinality]:
            diffs = indices_to_interval_sequence(comb)
            if any([is_rotation_equivalent(x, diffs) for x in transposition_equiv[cardinality]]):
                continue
            else:
                transposition_equiv[cardinality].append(diffs)

    assert sum([len(v) for v in transposition_equiv.values()]) == 352
    return transposition_equiv



In [ ]:
trans_eq = filter_transposition_equivalence(all_4096)
trans_eq

## <font color='purple'> Exercise 3: Inversion Equivalence

## <font color='crimson'> Solution 3

In [ ]:
def filter_inversion_equivalence(transposition_equiv: dict) -> dict:
    """
    Filter by introducing inversion equivalence, leading to 224 distinct chords.
    """
    assert sum([len(v) for v in transposition_equiv.values()]) == 352

    inversion_equiv = dict()
    inversion_equiv[0] = [None]
    inversion_equiv[1] = [None]
    for cardinality in range(2, 13):
        inversion_equiv[cardinality] = []
        for diffs in transposition_equiv[cardinality]:
            inversion = diffs[::-1]
            if any([is_rotation_equivalent(x, inversion) for x in inversion_equiv[cardinality]]):
                continue
            else:
                inversion_equiv[cardinality].append(diffs)

    assert sum([len(v) for v in inversion_equiv.values()]) == 224
    return inversion_equiv

In [ ]:
final = filter_inversion_equivalence(trans_eq)

In [ ]:
final

## <font color='Green'> Appendix

The above calculates on pitch class space on one octave.
For any one of the chords above, we could consider manifesting in different octaves.
The clearest demonstration is to consider one single pitch and all the possibilities there
(basically, that one pitch being present or absent in each octave).


## <font color='purple'> Ex(tra|er)cise: Octave permuatation

Modern pianos have 7 octaves from A0 (MIDI number **) up.

Calculate every permutation of just the note A across all these octaves.

## <font color='crimson'> Solution

In [ ]:
def get_all_octaves(
        pitch_class: int = 9,
        num_octaves: int = 7,
        convert_to_midi: bool = False,
) -> list:
    all_octaves = []

    for how_many_octaves in range(1, num_octaves + 1):
        all_octaves.extend(
            tuple(
                combinations(
                    range(7), how_many_octaves
                )
            )
        )

    return all_octaves

In [ ]:
ao = get_all_octaves()
ao

In [ ]:
def octaves_to_midi(
        octave_combinations: list,
        pitch_class: int = 9,
) -> list:
    out = []
    for x in octave_combinations:
        out.append([pitch_class + (12 * n) for n in x])
    return out

In [ ]:
midi_notes = octaves_to_midi(ao)

In [ ]:
len(midi_notes)

So there are 127 options for just the note A across 7 octaves.